# AgriTech market segmentation prototype
Use a consented pilot database, never synthetic farmers in production. Configure PGHOST, PGPORT, PGDATABASE, PGUSER, PGPASSWORD, MODEL_DIR and MLFLOW_TRACKING_URI first. Review segments with an agronomist before promotion. Expense transactions are not training features.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import pandas as pd
from app.store import Store
from app.engine import feature_rows
from app.training import train
store = Store()
profiles = store.profiles()
print(f'{len(profiles)} consented profiles')
features = pd.DataFrame(feature_rows(profiles))
features.describe(include='all') if not features.empty else 'Collect consented profiles first'

In [ ]:
reports = []
for algorithm in ['kmeans', 'kprototypes', 'hdbscan']:
    reports.append(store.with_cohort_lock(lambda rows: train(rows, algorithm)))
pd.DataFrame([{k: v for k, v in r.items() if k in ['status', 'algorithm', 'profiles', 'silhouette', 'clusterSizes', 'qualityGate']} for r in reports])

## Human validation
Inspect cluster sizes, noise and understandable segment names. Ask an agronomist or extension officer whether groups support the agreed decisions. Assess coverage for new and small farms. Choose an algorithm and retrain that candidate; only then run the documented promote command with the reviewer name. Numerical silhouette alone does not establish agronomic usefulness.